# TAWOS Data Extraction

Loads the official TAWOS MySQL dump (DOI 10.5522/04/21308124) into a MySQL server inside this Kaggle notebook and exports one CSV per project containing the issues that have story points.

**Input:** Kaggle dataset with `TAWOS.sql` (or `TAWOS.sql.zip`).
**Output (in `/kaggle/working`):**
- `csv/<PROJECT>.csv` — one per project: `project, issue_id, summary, description, created, storypoint, text`
- `all_issues_sp.csv` — everything in one file
- `manifest.csv` — per-project row counts + story point stats (verification table)
- `eda.png` — quick plots
- `tawos_clean_csv.zip` — all per-project CSVs zipped

After it finishes: **Save Version → Save & Run All**, then publish/use the output as your own Dataset.
Settings: CPU only (no GPU needed), Internet ON (apt install).

In [ ]:
import subprocess, time
def run(cmd, **kw):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, **kw)

# dump is from MySQL 8 (collation utf8mb4_0900_ai_ci breaks MariaDB) -> prefer mysql-server 8
run("apt-get update -qq")
r = run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mysql-server >/tmp/apt.log 2>&1",
        capture_output=True, text=True)
ENGINE = 'mysql' if r.returncode == 0 else 'mariadb'
if ENGINE == 'mariadb':
    print("mysql-server install failed -> mariadb fallback")
    run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mariadb-server >>/tmp/apt.log 2>&1")

# fast-load + big-packet tuning, then start
cnf = '[mysqld]\nmax_allowed_packet=512M\ninnodb_flush_log_at_trx_commit=2\ninnodb_buffer_pool_size=8G\nskip-log-bin\n'
open('/etc/mysql/conf.d/load.cnf','w').write(cnf)
run(f"service {ENGINE} start")
time.sleep(8)
ok = run("mysql -e 'SELECT VERSION()'", capture_output=True, text=True)
if ok.returncode != 0:
    print("service start failed, trying mysqld_safe")
    subprocess.Popen("mysqld_safe >/tmp/mysqld.log 2>&1 &", shell=True)
    time.sleep(20)
# TCP user for pandas (root is socket-auth on both engines)
run('mysql -e ' + repr("CREATE USER IF NOT EXISTS 'k'@'127.0.0.1' IDENTIFIED BY 'k'; GRANT ALL PRIVILEGES ON *.* TO 'k'@'127.0.0.1';"))
r = run("mysql -uk -pk -h 127.0.0.1 -e 'SELECT VERSION()'", capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, 'no working MySQL/MariaDB'

In [ ]:
import glob, os, zipfile
os.makedirs("/kaggle/tmp", exist_ok=True)
cands = glob.glob("/kaggle/input/**/*.sql*", recursive=True)
print("found in inputs:", cands)
for c in cands:
    if c.endswith(".zip"):
        with zipfile.ZipFile(c) as z:
            z.extractall("/kaggle/tmp")
sqls = glob.glob("/kaggle/tmp/**/*.sql", recursive=True) + [c for c in cands if c.endswith(".sql")]
assert sqls, "No .sql file found — is the TAWOS dataset attached?"
DUMP = max(sqls, key=os.path.getsize)
print("DUMP =", DUMP, "|", round(os.path.getsize(DUMP)/1e6, 1), "MB")

In [ ]:
run("mysql -e 'DROP DATABASE IF EXISTS tawos; CREATE DATABASE tawos'")
DUMPX = DUMP
if ENGINE == 'mariadb':  # rewrite MySQL8 collation MariaDB lacks; input is read-only -> copy first
    import shutil
    DUMPX = '/kaggle/tmp/TAWOS_fixed.sql'
    if not os.path.exists(DUMPX):
        shutil.copy(DUMP, DUMPX + '.tmp')
        run(f"sed -i 's/utf8mb4_0900_ai_ci/utf8mb4_general_ci/g' '{DUMPX}.tmp'")
        os.rename(DUMPX + '.tmp', DUMPX)
t0 = time.time()
r = subprocess.run(f"mysql --max_allowed_packet=512M tawos < '{DUMPX}'", shell=True, capture_output=True, text=True)
print("load rc:", r.returncode, "| minutes:", round((time.time()-t0)/60, 1))
if r.returncode != 0:
    print(r.stderr[-3000:])
r = run("mysql tawos -e 'SHOW TABLES'", capture_output=True, text=True)
print(r.stdout)
assert r.stdout.strip(), "DB empty after load"

In [ ]:
import subprocess as _sp
_sp.run("pip install -q pymysql", shell=True)
import pandas as pd, sqlalchemy
engine = sqlalchemy.create_engine("mysql+pymysql://k:k@127.0.0.1/tawos")

# MySQL 8 returns I_S columns uppercase -> alias + normalize
alltabs = pd.read_sql("SELECT TABLE_NAME AS tname, IFNULL(TABLE_ROWS,0) AS nrows FROM information_schema.tables WHERE table_schema='tawos'", engine).sort_values('nrows', ascending=False)
print(alltabs.to_string(index=False))

def cols(t):
    q = "SELECT COLUMN_NAME AS cname, DATA_TYPE AS dtype FROM information_schema.columns WHERE table_schema='tawos' AND LOWER(TABLE_NAME)=LOWER(%s)"
    d = pd.read_sql(q, engine, params=[(t,)])
    d.columns = ['column_name', 'data_type']
    return d

def real_name(lower):
    r = pd.read_sql("SELECT TABLE_NAME AS tname FROM information_schema.tables WHERE table_schema='tawos' AND LOWER(TABLE_NAME)=%s", engine, params=[(lower,)])
    return r.tname.iloc[0] if len(r) else None

ISSUE, PROJ = real_name('issue'), real_name('project')
print("ISSUE =", ISSUE, "| PROJ =", PROJ)

sp_hits = []
for t in alltabs.tname:
    c = cols(t)
    hits = [n for n in c.column_name if 'story' in str(n).lower()]
    if hits:
        sp_hits.append((t, hits))
print("story-point columns live in:", sp_hits)
for t in [ISSUE, PROJ]:
    print(t, "->", list(cols(t).column_name) if t else None)

In [ ]:
# ---- build extraction query from detected schema (TAWOS: Issue.Title/Description_Text/Story_Point, Project.Name) ----
def pick(clist, *pats, required=True):
    for pat in pats:                      # exact match wins
        for n in clist:
            if str(n).lower() == pat:
                return n
    for pat in pats:                      # then substring
        for n in clist:
            if pat in str(n).lower():
                return n
    if required:
        raise KeyError(f"no column matching {pats} in {list(clist)}")
    return None

ic, pc = cols(ISSUE), cols(PROJ)
summary_c  = pick(ic.column_name, 'title', 'summary')
desc_c     = pick(ic.column_name, 'description_text', 'description', required=False)
proj_fk    = pick(ic.column_name, 'project')
proj_name  = pick(pc.column_name, 'name', 'title')
created_c  = pick(ic.column_name, 'creation_date', 'created', 'created_at', required=False)
key_c      = pick(ic.column_name, 'issue_key', 'key', 'jira_id', required=False)
spflag_c   = pick(ic.column_name, 'story_point_changed', required=False)

sel = f"p.{proj_name} AS project, i.ID AS issue_id"
if key_c:    sel += f", i.{key_c} AS issue_key"
sel += f", i.{summary_c} AS summary"
if desc_c:    sel += f", i.{desc_c} AS description"
if created_c: sel += f", i.{created_c} AS created"

if sp_hits and sp_hits[0][0] != ISSUE:
    SP_TABLE = sp_hits[0][0]
    sc = cols(SP_TABLE)
    sp_fk  = pick(sc.column_name, 'issue')
    sp_val = pick(sc.column_name, 'value', 'point', 'story')
    sel += f", s.{sp_val} AS storypoint"
    q = f"SELECT {sel} FROM {ISSUE} i JOIN {PROJ} p ON i.{proj_fk}=p.ID JOIN {SP_TABLE} s ON s.{sp_fk}=i.ID"
else:
    sp_val = pick(ic.column_name, 'story_point', 'story', 'estimate')
    sel += f", i.{sp_val} AS storypoint"
    if spflag_c: sel += f", i.{spflag_c} AS sp_changed_after_est"
    q = f"SELECT {sel} FROM {ISSUE} i JOIN {PROJ} p ON i.{proj_fk}=p.ID"

print("QUERY: " + q)
df = pd.read_sql(q, engine)
print("rows:", df.shape)

In [ ]:
# ---- clean ----
import re, numpy as np
n0 = len(df)
df['storypoint'] = pd.to_numeric(df['storypoint'], errors='coerce')
df = df.dropna(subset=['storypoint']).drop_duplicates('issue_id')

def clean_text(x):
    if not isinstance(x, str): return ""
    x = re.sub(r'<[^>]+>', ' ', x)
    return re.sub(r'\s+', ' ', x).strip()

for c in ['summary', 'description']:
    if c in df: df[c] = df[c].fillna("").map(clean_text)
df['text'] = (df['summary'] + (" " + df['description'] if 'description' in df else "")).str.strip()
df['storypoint'] = df['storypoint'].astype(float)
df = df.reset_index(drop=True)
print(f"{n0} -> {len(df)} rows (dropped null/dup storypoints)")
df.head(3)

In [ ]:
# ---- export: per-project CSVs + manifest + zip ----
import shutil
os.makedirs("/kaggle/working/csv", exist_ok=True)
rows = []
for proj, g in df.groupby('project'):
    safe = re.sub(r'[^A-Za-z0-9_]', '_', str(proj))
    g.to_csv(f"/kaggle/working/csv/{safe}.csv", index=False)
    rows.append(dict(project=proj, file=f"{safe}.csv", n=len(g),
                     sp_min=g.storypoint.min(), sp_median=g.storypoint.median(),
                     sp_max=g.storypoint.max(), sp_mean=round(g.storypoint.mean(), 2)))
man = pd.DataFrame(rows).sort_values('n', ascending=False)
man.to_csv("/kaggle/working/manifest.csv", index=False)
df.to_csv("/kaggle/working/all_issues_sp.csv", index=False)
print(man.to_string(index=False))
shutil.make_archive("/kaggle/working/tawos_clean_csv", "zip", "/kaggle/working/csv")
print("\nDONE. Files in /kaggle/working:", os.listdir('/kaggle/working'))

In [ ]:
# ---- EDA ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
man.head(35).plot.bar(x='project', y='n', ax=ax[0], legend=False)
ax[0].set_title('Issues with story points per project')
ax[0].set_ylabel('count')
for p in man.project.head(6):
    df[df.project == p].storypoint.hist(bins=30, alpha=0.5, ax=ax[1], label=str(p)[:12])
ax[1].set_title('Story point distribution (top 6 projects)')
ax[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/eda.png', dpi=120)
plt.show()

## After the run
1. **Save Version → Save & Run All (commit)** — output becomes a stable Dataset.
2. Verify `manifest.csv` row counts against Tawosi et al. published per-project numbers (abecoup/story-point-estimation repo = reference).
3. Attach this output dataset to every later notebook (embeddings, CNN, 39-pair runs).
4. Commit this notebook and `manifest.csv` to the project repo.